# LLM-based synaptic type classification (excitatory / inhibitory)

This notebook:
1. Loads `Metadata_Neurons(manual).csv` with neuron metadata from the Drosophila larval L1 EM connectome.
2. For each neuron, calls Claude (Anthropic) with a detailed prompt to classify the neuron as **excitatory** or **inhibitory** based on name, morphology metrics, and CATMAID annotations.
3. Output is strictly a single word per neuron: `excitatory` or `inhibitory`.
4. Results are collected into a DataFrame with columns: `neuron_id`, `synapse_type`.

## Imports

In [1]:
import json
import re
import time
import pandas as pd
import anthropic
from tqdm.notebook import tqdm
from collections import Counter
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir("../..")

# Sorry for this shitty path variables, It will be fixed on server soon (fix for this error: "Permission denied: '/root/.cargo/bin'")
os.environ["PATH"] = ":".join(
    p for p in os.environ["PATH"].split(":")
    if not p.startswith("/root")
)


import pymaid

catmaid_url = 'https://l1em.catmaid.virtualflybrain.org'
http_user = None
http_password = None
project_id = 1

rm = pymaid.CatmaidInstance(catmaid_url, http_user, http_password, project_id)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


## Load neuron metadata (3k via CATMAID)

In [2]:
skids_3k = sorted(map(int, pymaid.get_skids_by_annotation("mw brain and inputs")))

csv_path = "./Datasets/Generated/Metadata_Neurons(manual).csv"
df = pd.read_csv(csv_path)
df = df.fillna("")
cols = ["neuron_id", "name", "n_nodes", "n_connectors", "n_branches", "n_leafs", "cable_length", "annotation"]
df = df[df["neuron_id"].isin(skids_3k)][cols].copy()

df

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


,neuron_id,name,n_nodes,n_connectors,n_branches,n_leafs,cable_length,annotation
0,7766016,H-shaped _a1l,5745,374,138,141,620329.900,"['YYCWMLEH', '4_level-4_clusterID-22_signal-fl..."
2,15564807,AN-R-Sens-B1-AVa-22,800,34,7,8,69523.360,"['monosynaptic to IPC-like and Se0', 'Miroschn..."
3,17383431,BAlp_ant ascending dendrite left,896,109,64,70,205249.700,"['1_level-1_clusterID-1_signal-flow_-0.067', '..."
5,14434314,S1: neuron left,2682,208,133,138,339644.500,"['5_level-3_clusterID-14_signal-flow_-0.319', ..."
6,11714576,Weird PN to LH Right,3430,209,136,142,382807.470,"['1_level-3_clusterID-12_signal-flow_0.836', '..."
...,...,...,...,...,...,...,...,...
5006,15564782,AN-R-Sens-B1-AVa-23,937,37,15,16,89306.234,"['monosynaptic to IPC-like and Se0', 'Miroschn..."
5007,8142831,BLP12 bilateral horseshoe 2 left,2192,119,66,67,267116.700,"['26_level-5_clusterID-37_signal-flow_-0.688',..."
5010,3882998,CP contra to SEZ left; MB1: incomplete neuron ...,2528,58,33,34,298192.720,"['mw dSEZ', '8_level-4_clusterID-16_signal-flo..."
5011,15679484,MN-L-Sens-B2-VM-08,851,30,5,6,70720.140,"['Miroschnikow et al inputs and outputs', 'AM_..."


## API key

In [4]:
ANTHROPIC_API_KEY = ""

## System prompt: background, data, task, output format

In [5]:
SYSTEM_PROMPT = """
You are an expert in Drosophila neuroanatomy and connectomics. Your task is to classify individual neurons from the first-instar Drosophila larval brain connectome as either **excitatory** or **inhibitory** based on all available metadata (name, morphology, and annotations). You must output exactly one word: excitatory or inhibitory—no reasoning, no explanation, no other text.

## Background and data source

**Connectome and dataset:** The data come from the complete synaptic-resolution connectome of the Drosophila melanogaster first-instar larval brain (Winding, Pedigo et al., Science 2023, doi:10.1126/science.add9330). This connectome comprises 3016 neurons and ~548,000 synapses. Neurons and synapses were reconstructed in a nanometer-resolution electron microscopy (EM) volume of the central nervous system (CNS) using CATMAID (Collaborative Annotation Toolkit for Massive Amounts of Image Data)—an open-source tool for collaborative reconstruction and annotation of neuronal circuits from EM data (catmaid.readthedocs.io). The larval brain dataset is hosted and described as the L1 EM Catmaid Dataset on Virtual Fly Brain (VFB; see virtualflybrain.org blog/releases/catmaid/l1em/). It was produced by Albert Cardona's lab at HHMI Janelia and collaborators. The same dataset and naming/annotation conventions are used in the Science 2023 paper and in public CATMAID/L1EM resources.

**Neuron types and brain organization:** In the larval brain, neurons were clustered into 93 connectivity-based types; many cell classes are known (sensory neurons, projection neurons, Kenyon cells, MBONs, MBINs/DANs, local neurons, descending neurons, etc.). In Drosophila, neurotransmitter identity and thus excitatory vs inhibitory character are often linked to brain region and cell type: e.g. cholinergic neurons are typically excitatory; GABAergic and certain other populations are inhibitory. Brain regions and neuron naming conventions often reflect anatomy (neuropil, hemisphere, lineage), and neurons in the same region or with similar names often share functional and neurotransmitter profiles. You may use your knowledge of Drosophila neurobiology, published connectome papers (including the Science 2023 paper and related L1/larval literature), and, if helpful, general neurobiological principles or online sources to inform your classification—but you must base your final answer on the specific fields provided for each neuron.

## Reference material (from the official sources below—use this when classifying)

**1) Science paper — The connectome of an insect brain (doi:10.1126/science.add9330):**
Complete synaptic-resolution connectome of the Drosophila larva brain: 3016 neurons, ~548,000 synapses. Reconstruction was done in a nanometer-resolution EM volume of the CNS using CATMAID. Neurons were clustered into 93 connectivity-based types; morphology within clusters was similar and known functions matched clusters (e.g. olfactory PNs, KCs, MBINs/MBONs). Brain inputs: sensory neurons (SNs) and ascending neurons (ANs). Outputs: ring gland neurons (RGNs), DNs to SEZ (DNsSEZ), DNs to VNC (DNsVNC). Four connection types: axo-dendritic (a-d, majority), axo-axonic (a-a), dendro-dendritic (d-d), dendro-axonic (d-a). In adult Drosophila, a-a connections between otherwise excitatory (cholinergic) KCs can be inhibitory due to inhibitory mAChR-B in axon terminals; cholinergic neurons are typically excitatory. DANs (dopaminergic), MBONs, MBINs, KCs, local neurons (LNs), projection neurons (PNs)—naming and region help infer neurotransmitter and thus excitatory vs inhibitory.

**2) CATMAID documentation (catmaid.readthedocs.io):**
CATMAID = Collaborative Annotation Toolkit for Massive Amounts of Image Data. Features: fast terabyte-scale image browsing; collaborative microcircuit reconstruction and annotation; flexible hierarchical semantic annotation; multiple linked image stack display; Neuron Catalog; SVG and WebGL-based neuronal morphology viewer; open source (GPLv3). Annotations are free-text or hierarchical tags added by users during tracing; they document what tracers observe (e.g. neuron type, region, synapse character, notes). Use annotations as direct but uncurated evidence from expert tracers.

**3) L1 EM Catmaid Dataset on Virtual Fly Brain (virtualflybrain.org/blog/releases/catmaid/l1em/):**
Drosophila Larval (L1) CNS tracing by Albert Cardona's lab at HHMI Janelia Research Campus and collaborators. Project: "L1 CNS"; stack: "L1 CNS 0-tilt" — L1 CNS ssTEM at 3.8×3.8×50 nm of whole Drosophila 1st instar larval CNS. This is the same dataset as in the Science 2023 connectome paper; neuron names and annotations follow the same conventions used in the VFB L1EM CATMAID server and API.

## Input fields (use all except neuron_id)

You will receive for each neuron:
- **name**: The neuron's name or label. Names often encode location, lineage, or type (e.g. brain region, hemisphere, neuropil, cell class). Pay close attention to this: in the Drosophila larval brain, naming conventions and anatomical location are strong predictors of cell type, and neurons in specific regions (e.g. mushroom body, lateral horn, antennal lobe, descending pathways) tend to cluster by neurotransmitter and thus excitatory vs inhibitory role. Use the name as a primary cue.
- **n_nodes**: Number of nodes (segments) in the neuron's morphology.
- **n_connectors**: Number of synaptic connectors (synapses) associated with this neuron.
- **n_branches**: Number of branches in the arbor.
- **n_leafs**: Number of leaf (terminal) segments.
- **cable_length**: Total cable length of the neuron (in the same units as in the dataset).
- **annotation**: A collection of free-text annotations added by human tracers during CATMAID-based reconstruction and proofreading. These annotations are the direct translation of what experienced biologists wrote while inspecting EM images. They may include: technical or workflow tags (e.g. cluster IDs, signal-flow values, "published", "paired"); anatomical or functional notes; references to brain regions, pathways, or papers; and sometimes explicit or implicit hints about synapse type, neurotransmitter, or neuron class (e.g. "cholinergic", "GABAergic", "inhibitory", "excitatory", receptor names, or morphological descriptors). The annotation field is uncurated and may contain noise, typos, or irrelevant tags—but it can also carry critical information about the neuron's type. Weigh annotation content carefully: treat explicit neurotransmitter or synapse-type mentions as strong cues; treat technical IDs and generic tags as weak or neutral; aggregate all cues together with name and morphology.

## How to decide

Consider the **whole picture**: name (location/type), morphology (n_nodes, n_connectors, n_branches, n_leafs, cable_length), and annotation text. Aggregate all factors together—do not rely on a single cue. Give the most weight to **name** (because it usually encodes position and type, and brain regions in the fly larva often group neurons of similar synaptic character) and to **annotation** (because it reflects direct observer notes from EM, including possible synapse-type or neurotransmitter hints). Account for possible annotation noise and tracer errors: annotations are unverified; they may contain typos, technical jargon, or irrelevant tags—weigh explicit neurotransmitter/synapse-type mentions highly and discount inconsistent or noisy tags. Prefer consistent or biologically meaningful signals over single ambiguous tokens. If you would need to look up a brain region or cell type to be more accurate, you may do so conceptually (or via your training) to map names/annotations to excitatory vs inhibitory; your answer must still be based only on the given fields and general Drosophila/larval connectome knowledge.

## Output format (strict)

You must respond with **exactly one word**, with no other characters, spaces, or punctuation before or after. The only two allowed answers are:
- excitatory
- inhibitory

Do not output reasoning, explanations, confidence, or alternative answers. Do not use "Excitatory" or "Inhibitory" (capitalized) or any other variant. Only one of the two words above, in lowercase.
"""

## User prompt template and classification function

In [6]:
USER_PROMPT_TEMPLATE = """
Classify this neuron. Output only one word: excitatory or inhibitory.

name: {name}
n_nodes: {n_nodes}
n_connectors: {n_connectors}
n_branches: {n_branches}
n_leafs: {n_leafs}
cable_length: {cable_length}
annotation: {annotation}
"""


def parse_synapse_type(raw: str) -> str:
    """Extract exactly 'excitatory' or 'inhibitory' from model output."""
    s = raw.strip().lower()
    if "excitatory" in s and "inhibitory" not in s:
        return "excitatory"
    if "inhibitory" in s:
        return "inhibitory"
    if s in ("excitatory", "inhibitory"):
        return s
    if s.startswith("excitatory"):
        return "excitatory"
    if s.startswith("inhibitory"):
        return "inhibitory"
    return ""  # unknown, caller may retry or mark missing


def call_claude_for_synapse_type(
    name: str,
    n_nodes: str,
    n_connectors: str,
    n_branches: str,
    n_leafs: str,
    cable_length: str,
    annotation: str,
    api_key: str,
) -> str:
    """Call Claude and return 'excitatory' or 'inhibitory'."""
    if not api_key:
        return ""
    client = anthropic.Anthropic(api_key=api_key)
    msg = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=64,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(
                    name=name or "",
                    n_nodes=n_nodes if n_nodes != "" else "",
                    n_connectors=n_connectors if n_connectors != "" else "",
                    n_branches=n_branches if n_branches != "" else "",
                    n_leafs=n_leafs if n_leafs != "" else "",
                    cable_length=cable_length if cable_length != "" else "",
                    annotation=annotation if annotation != "" else "(none)",
                ),
            }
        ],
    )
    raw = msg.content[0].text if msg.content else ""
    return parse_synapse_type(raw)


def safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()

## Run classification for all neurons

In [7]:
results = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Synapse type classification"):
    neuron_id = row["neuron_id"]
    name = safe_str(row["name"])
    n_nodes = safe_str(row["n_nodes"])
    n_connectors = safe_str(row["n_connectors"])
    n_branches = safe_str(row["n_branches"])
    n_leafs = safe_str(row["n_leafs"])
    cable_length = safe_str(row["cable_length"])
    annotation = safe_str(row["annotation"])

    synapse_type = call_claude_for_synapse_type(
        name=name,
        n_nodes=n_nodes,
        n_connectors=n_connectors,
        n_branches=n_branches,
        n_leafs=n_leafs,
        cable_length=cable_length,
        annotation=annotation,
        api_key=ANTHROPIC_API_KEY,
    )

    print(synapse_type)

    if not synapse_type:
        synapse_type = ""  # leave empty or set to a placeholder if you prefer

    results.append({"neuron_id": neuron_id, "synapse_type": synapse_type})

result_df = pd.DataFrame(results)
result_df

Synapse type classification:   0%|          | 0/5013 [00:00<?, ?it/s]

excitatory
inhibitory
excitatory
inhibitory



KeyboardInterrupt



## Summary and optional export

In [ ]:
print(result_df["synapse_type"].value_counts(dropna=False))
missing = (result_df["synapse_type"] == "").sum()
if missing:
    print(f"Missing/unknown: {missing}")

# Optional: save to CSV
# result_df.to_csv("../../Datasets/Generated/LLM_synapse_type_classification.csv", index=False)